# Page 13: does GRPO teach the model to re-check, and do the re-checks help?
### Qwen2.5-1.5B-Instruct, before and across 400 steps of GRPO on GSM8K

| § | Step |
|---|---|
| 1 | Sample: every checkpoint writes 8 chains for each of the 600 GSM8K test problems the two-axis run used |
| 2 | Mark: find re-check language in every chain (two families: *checks* and *revisions*) |
| 3 | Measure: how often each checkpoint re-checks, and whether chains that re-check are right more often than chains on the same problem that don't |
| 4 | Specimens: three step-400 chains with a revision, drawn at random |
| 5 | Results file: every number in one JSON |

**Checkpoints.** Step 0 is `Qwen/Qwen2.5-1.5B-Instruct` from the Hub. Steps 25 to 400 are the full fine-tune checkpoints the two-axis GRPO notebook saved to Google Drive (`CKPT_DIR`).

**Sampling.** Identical to the two-axis evaluation: same system prompt, same `#### n` extractor, same 600 test problems (seed 42), temperature 0.7, top-p 0.95, up to 1,024 tokens.

**Same-problem comparison.** Hard problems invite both re-checking and wrong answers, so comparing all chains with all chains would blame the re-check for the problem. §3 compares chains *within* a problem: for every problem where some chains re-check and some don't, it takes the difference in accuracy, then averages over those problems, with a bootstrap interval over problems.

Every run is a full run in its own `run-<UTC>` folder. A run interrupted by a disconnect resumes where it stopped; a finished run (one with `derived.json`) is never read again.

## 0. Environment

In [ ]:
import os, sys, subprocess
MARK = "/content/.rechecking_env_ok"

def pip(*args):
    print(">", "pip", *args); subprocess.run([sys.executable, "-m", "pip", *args], check=True)

if not os.path.exists(MARK):
    pip("install", "-q", "--upgrade", "pip")
    pip("uninstall", "-y", "-q", "torchvision", "torchaudio")        # Colab's copies will mismatch vLLM's torch
    pip("install", "-q", "--upgrade", "vllm>=0.8.5,<0.33.0")         # brings its own torch + nvidia-* libs
    pip("install", "-q", "--upgrade", "datasets", "pandas", "matplotlib")
    chk = subprocess.run([sys.executable, "-c", "import torch, vllm; print(torch.__version__, vllm.__version__)"],
                         capture_output=True, text=True)
    if chk.returncode != 0:
        print(chk.stderr[-3000:]); raise SystemExit("environment check failed, see error above")
    print("versions:", chk.stdout.strip())
    open(MARK, "w").write("ok")
    print("\nEnvironment ready. Restarting the runtime now (expected). Then run all cells from the top.")
    os.kill(os.getpid(), 9)
else:
    print("environment already prepared, skipping install")

In [ ]:
import os, sys, json, re, shlex, subprocess, random, time, hashlib
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
import vllm

gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | VRAM {round(gpu.total_memory / 2**30)} GB | torch {torch.__version__} | vllm {vllm.__version__}")
assert torch.cuda.is_available()

### 0.1 Knobs, the only cell you edit
`CKPT_DIR` is where the two-axis notebook saved its checkpoints (`step25`, `step50`, ... `step400`). The sampling settings match that run's evaluation and should stay as they are.

In [ ]:
MODEL_ID     = "Qwen/Qwen2.5-1.5B-Instruct"     # step 0, and the tokenizer for every checkpoint
CKPT_DIR     = "/content/drive/MyDrive/two_axis/1.5B_A100-80GB_seed42/ckpt"
STEPS        = [0, 25, 50, 100, 200, 400]

SEED         = 42          # picks the 600 test problems and seeds sampling, as in the two-axis run
N_EVAL       = 600
K            = 8           # chains per problem per checkpoint
TEMPERATURE  = 0.7
TOP_P        = 0.95
MAX_TOKENS   = 1024
GPU_UTIL     = 0.85

N_SPECIMENS   = 3          # step-400 chains with a revision, shown on the page
SPECIMEN_SEED = 7
N_BOOT        = 2000       # bootstrap resamples over problems

RESULTS_ROOT = "/content/drive/MyDrive/page13_rechecking"

from google.colab import drive; drive.mount("/content/drive")
missing = [s for s in STEPS if s and not (Path(CKPT_DIR) / f"step{s}" / "config.json").exists()]
assert not missing, f"no checkpoint folder with config.json for steps {missing} under {CKPT_DIR}"

# one folder per run; resume the newest unfinished one, never reopen a finished one
Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)
open_runs = sorted(p for p in Path(RESULTS_ROOT).glob("run-*") if not (p / "derived.json").exists())
RUN_DIR = open_runs[-1] if open_runs else Path(RESULTS_ROOT) / f"run-{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}"
(RUN_DIR / "samples").mkdir(parents=True, exist_ok=True)
SETTINGS = dict(model_id=MODEL_ID, ckpt_dir=CKPT_DIR, steps=STEPS, seed=SEED, n_eval=N_EVAL, k=K,
                temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS,
                n_specimens=N_SPECIMENS, specimen_seed=SPECIMEN_SEED, n_boot=N_BOOT)
if (RUN_DIR / "settings.json").exists():
    assert json.load(open(RUN_DIR / "settings.json")) == SETTINGS, "knobs changed since this run started; start a new run"
else:
    json.dump(SETTINGS, open(RUN_DIR / "settings.json", "w"), indent=1)
print(("resuming " if open_runs else "new run ") + str(RUN_DIR))

### 0.2 Shared module and the sampling worker
`common.py` is the two-axis notebook's prompt, extractor and test-set loader, unchanged. `sample.py` loads one checkpoint in vLLM, writes `K` chains per problem, and saves every chain's text.

In [ ]:
%%writefile common.py
import re
from datasets import load_dataset

SYSTEM = "Think step by step. Then give the final answer on its own line as: #### <number>"
NUM = r"-?\$?\d[\d,]*\.?\d*"

def normalize(s):
    if s is None: return None
    s = s.replace(",", "").replace("$", "").strip().rstrip(".")
    try:
        f = float(s); return str(int(f)) if f == int(f) else str(f)
    except ValueError:
        return None

def extract(text):
    """Last '#### n' if present, else the last number in the text (what a truncated trace yields)."""
    m = re.findall(r"####\s*(" + NUM + ")", text)
    if m: return normalize(m[-1])
    nums = re.findall(NUM, text)
    return normalize(nums[-1]) if nums else None

def gold(answer_field): return normalize(answer_field.split("####")[-1])

def to_prompt(ex):
    return {"prompt": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": ex["question"]}],
            "answer": gold(ex["answer"])}

def load_gsm8k(split, n=None, seed=0):
    ds = load_dataset("openai/gsm8k", "main", split=split).map(to_prompt).shuffle(seed=seed)
    return ds.select(range(n)) if n else ds

In [ ]:
%%writefile sample.py
# one checkpoint -> K chains per test problem, every text saved
import argparse, json, os, time
from vllm import LLM, SamplingParams
from common import load_gsm8k, extract
p = argparse.ArgumentParser()
for f in ["--model", "--tokenizer", "--out"]: p.add_argument(f, required=True)
for f in ["--n", "--k", "--max_tokens", "--seed"]: p.add_argument(f, type=int, required=True)
for f in ["--temperature", "--top_p", "--gpu_util"]: p.add_argument(f, type=float, required=True)
a = p.parse_args(); t0 = time.time()
ds = load_gsm8k("test", n=a.n, seed=a.seed)
llm = LLM(model=a.model, tokenizer=a.tokenizer, dtype="bfloat16", seed=a.seed, gpu_memory_utilization=a.gpu_util,
          max_model_len=512 + a.max_tokens, enable_prefix_caching=True)
tok = llm.get_tokenizer()
prompts = [tok.apply_chat_template(ex["prompt"], tokenize=False, add_generation_prompt=True) for ex in ds]
outs = llm.generate(prompts, SamplingParams(n=a.k, temperature=a.temperature, top_p=a.top_p, max_tokens=a.max_tokens, seed=a.seed))
rows = []
for i, (ex, out) in enumerate(zip(ds, outs)):
    rows.append({"i": i, "question": ex["question"], "gold": ex["answer"],
                 "chains": [{"text": s.text, "answer": extract(s.text), "len": len(s.token_ids),
                             "truncated": len(s.token_ids) >= a.max_tokens} for s in out.outputs]})
tmp = a.out + ".tmp"
json.dump(rows, open(tmp, "w")); os.replace(tmp, a.out)
n = sum(len(r["chains"]) for r in rows)
acc = sum(c["answer"] == r["gold"] for r in rows for c in r["chains"]) / n
print(f"done: {len(rows)} problems x {a.k} chains, accuracy {acc:.3f}, {time.time() - t0:.0f}s")

## 1. Sample
One subprocess per checkpoint, so vLLM's memory is freed between them. A checkpoint already sampled in this run is skipped.

In [ ]:
def run(cmd):
    print(">", cmd, flush=True)
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in proc.stdout:
        tail = (tail + [line])[-60:]
        if any(t in line for t in ("done:", "Error", "error")): print(line.rstrip()[-220:], flush=True)
    proc.wait()
    if proc.returncode != 0:
        print("".join(tail)); raise RuntimeError(f"worker failed (exit {proc.returncode})")

for step in STEPS:
    out = RUN_DIR / "samples" / f"step{step}.json"
    if out.exists():
        print(f"step {step}: sampled earlier in this run"); continue
    model = MODEL_ID if step == 0 else f"{CKPT_DIR}/step{step}"
    run(f"{sys.executable} sample.py --model {model} --tokenizer {MODEL_ID} --out {out} --n {N_EVAL} --k {K} "
        f"--max_tokens {MAX_TOKENS} --seed {SEED} --temperature {TEMPERATURE} --top_p {TOP_P} --gpu_util {GPU_UTIL}")
SAMPLES = {s: json.load(open(RUN_DIR / "samples" / f"step{s}.json")) for s in STEPS}

## 2. Mark
Two families of re-check language, matched case-insensitively in the chain only (never the question):

- **check**: the chain announces a verification (*let's check*, *verify*, *double-check*).
- **revision**: the chain doubts or corrects itself (*wait*, *actually*, *mistake*, *recalculate*, *let me redo*).

A chain that writes two different `#### n` lines has **changed its final answer**, the plainest sign of a correction. The audit cell prints random matches in context, so the lexicon can be read, not trusted.

In [ ]:
MARKERS = {
    "check": [
        r"\bdouble[- ]?check",
        r"\bre-?check",
        r"\bverif(?:y|ying|ied|ies|ication)\b",
        r"\b(?:let me|let's|let us|we can|we should|i will|i'll)\s+(?:quickly\s+|now\s+)?check\b",
        r"\bcheck(?:ing)?\s+(?:our|my|the|this)\s+(?:work|answer|result|calculation|solution)",
    ],
    "revision": [
        r"\bwait\b",
        r"\bhmm+\b",
        r"\boops\b",
        r"\bactually\b",
        r"\b(?:i|we)\s+made\s+(?:a|an)\s+(?:mistake|error)",
        r"\bmistake\b",
        r"\bmiscalculat\w*",
        r"\bre-?(?:calculate|calculating|evaluate|evaluating|examine|consider|do)\b",
        r"\blet(?:\s+me|'s|\s+us)\s+(?:try again|correct|redo|re-?\w+)",
        r"\bthat(?:'s| is)\s+(?:not right|wrong|incorrect)\b",
    ],
}
COMPILED = {fam: [re.compile(p, re.I) for p in pats] for fam, pats in MARKERS.items()}
FINAL = re.compile(r"####\s*(-?\$?\d[\d,]*\.?\d*)")

def mark(text):
    hits = {fam: [(m.start(), m.end(), p.pattern) for p in pats for m in p.finditer(text)] for fam, pats in COMPILED.items()}
    finals = {f.replace(",", "").replace("$", "").rstrip(".") for f in FINAL.findall(text)}
    return hits, len(finals) > 1

rows = []
for step, probs in SAMPLES.items():
    for pr in probs:
        for j, c in enumerate(pr["chains"]):
            hits, changed = mark(c["text"])
            rows.append(dict(step=step, i=pr["i"], j=j, correct=c["answer"] == pr["gold"], len=c["len"],
                             truncated=c["truncated"], check=bool(hits["check"]), revision=bool(hits["revision"]),
                             changed=changed, **{f"pat:{p}": any(h[2] == p for h in hits[fam])
                                                  for fam, pats in MARKERS.items() for p in pats}))
DF = pd.DataFrame(rows); DF["any"] = DF.check | DF.revision
print(DF.groupby("step")[["check", "revision", "any", "changed", "correct"]].mean().round(3))

In [ ]:
# audit: random matches in context, per family, at the first and last step
rng = random.Random(SEED)
for step in (STEPS[0], STEPS[-1]):
    for fam in MARKERS:
        pool = [(pr, c) for pr in SAMPLES[step] for c in pr["chains"] if mark(c["text"])[0][fam]]
        print(f"\n=== step {step} · {fam} · {len(pool)} chains ===")
        for pr, c in rng.sample(pool, min(6, len(pool))):
            s, e, pat = mark(c["text"])[0][fam][0]
            ctx = c["text"][max(0, s - 90): e + 90].replace("\n", " ")
            print(f"[{'right' if c['answer'] == pr['gold'] else 'wrong'}] ...{ctx}...")

## 3. Measure
Per checkpoint: how often chains re-check, accuracy with and without, and the same-problem difference with a 95% bootstrap interval over problems.

In [ ]:
def same_problem(d, col, rng):
    # problems with both kinds of chain; difference in accuracy, averaged over those problems
    g = d.groupby(["i", col]).correct.mean().unstack(col)
    if True not in g or False not in g: return dict(n_problems=0, diff=None, lo=None, hi=None)
    g = g.dropna()
    diffs = (g[True] - g[False]).to_numpy()
    if len(diffs) == 0: return dict(n_problems=0, diff=None, lo=None, hi=None)
    boot = [rng.choice(diffs, len(diffs)).mean() for _ in range(N_BOOT)]
    return dict(n_problems=int(len(diffs)), diff=float(diffs.mean()),
                lo=float(np.percentile(boot, 2.5)), hi=float(np.percentile(boot, 97.5)))

PER_STEP = []
for step in STEPS:
    d = DF[DF.step == step]; rng = np.random.default_rng(SEED)
    rec = dict(step=step, n_chains=int(len(d)), accuracy=float(d.correct.mean()), mean_len=float(d.len.mean()),
               frac_truncated=float(d.truncated.mean()), changed_final=float(d.changed.mean()))
    for fam in ("check", "revision", "any"):
        w, wo = d[d[fam]], d[~d[fam]]
        rec[fam] = dict(rate=float(d[fam].mean()),
                        acc_with=float(w.correct.mean()) if len(w) else None,
                        acc_without=float(wo.correct.mean()) if len(wo) else None,
                        len_with=float(w.len.mean()) if len(w) else None,
                        len_without=float(wo.len.mean()) if len(wo) else None,
                        same_problem=same_problem(d, fam, rng))
    rec["patterns"] = {c[4:]: float(d[c].mean()) for c in d.columns if c.startswith("pat:")}
    PER_STEP.append(rec)

show = pd.DataFrame([{ "step": r["step"], "acc": r["accuracy"], "len": r["mean_len"],
    **{f"{f} rate": r[f]["rate"] for f in ("check", "revision")},
    **{f"{f} Δ same-problem": r[f]["same_problem"]["diff"] for f in ("check", "revision")},
    **{f"{f} 95% CI": (r[f]["same_problem"]["lo"], r[f]["same_problem"]["hi"]) for f in ("check", "revision")},
    "changed final": r["changed_final"]} for r in PER_STEP]).set_index("step")
display(show.round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for fam, col in (("check", "tab:blue"), ("revision", "tab:red")):
    ax[0].plot(STEPS, [100 * r[fam]["rate"] for r in PER_STEP], "o-", color=col, label=fam)
    sp = [r[fam]["same_problem"] for r in PER_STEP]
    ok = [i for i, s in enumerate(sp) if s["diff"] is not None]
    ax[1].errorbar([STEPS[i] for i in ok], [100 * sp[i]["diff"] for i in ok],
                   yerr=[[100 * (sp[i]["diff"] - sp[i]["lo"]) for i in ok], [100 * (sp[i]["hi"] - sp[i]["diff"]) for i in ok]],
                   fmt="o-", color=col, capsize=3, label=fam)
ax[0].set(xscale="symlog", xlabel="RL step", ylabel="chains with the marker (%)", title="How often the model re-checks")
ax[1].axhline(0, color="grey", lw=0.8)
ax[1].set(xscale="symlog", xlabel="RL step", ylabel="accuracy, with minus without (points)", title="Same problem: does re-checking help?")
for a in ax: a.legend(); a.set_xticks(STEPS); a.set_xticklabels(STEPS)
plt.tight_layout(); plt.show()

## 4. Specimens
Drawn at random (seed `SPECIMEN_SEED`) from one stated pool: every step-400 chain with a revision marker. Right and wrong chains are drawn alike.

In [ ]:
last = STEPS[-1]
pool = [(pr, j, c) for pr in SAMPLES[last] for j, c in enumerate(pr["chains"]) if mark(c["text"])[0]["revision"]]
picked = random.Random(SPECIMEN_SEED).sample(pool, min(N_SPECIMENS, len(pool)))
SPECIMENS = []
for pr, j, c in picked:
    hits, changed = mark(c["text"])
    SPECIMENS.append(dict(step=last, problem=pr["i"], chain=j, question=pr["question"], gold=pr["gold"],
                          answer=c["answer"], correct=c["answer"] == pr["gold"], len=c["len"], changed_final=changed,
                          text=c["text"], marks=sorted([[s, e, fam] for fam in hits for s, e, _ in hits[fam]])))
    print(f"\n### problem {pr['i']} · chain {j} · {'right' if c['answer'] == pr['gold'] else 'wrong'} · {c['len']} tokens\n{c['text']}")
print(f"\npool: {len(pool)} step-{last} chains with a revision marker")

## 5. Results file
`derived.json` closes the run. Copy `page-13-rechecking.json` into the site's `web/data/`.

In [ ]:
DERIVED = dict(settings=SETTINGS, markers=MARKERS, versions=dict(torch=torch.__version__, vllm=vllm.__version__, gpu=gpu.name),
               run_dir=str(RUN_DIR), finished_utc=datetime.now(timezone.utc).isoformat(timespec="seconds"),
               per_step=PER_STEP, specimens=SPECIMENS, specimen_pool=len(pool))
json.dump(DERIVED, open(RUN_DIR / "derived.json", "w"), indent=1)
json.dump(DERIVED, open(RUN_DIR / "page-13-rechecking.json", "w"), indent=1)
print("wrote", RUN_DIR / "derived.json", "and page-13-rechecking.json")